# Formula.vLLM — reported vs reproduced gaps

Reads `data/pr_<n>/results.<hw-slug>.json` (written by `pipeline/record.py` → `derive.py`) through `stats/build_stats.py`, the same code that writes `stats/prs.csv`, `fields.csv`, `status.csv` and `gaps.png` (run by `run.sh --reproduce`). **gap = PR's solution − baseline**, taken relative to its own baseline so units cancel
(`(baseline − PR's solution) ÷ baseline`; throughput units flipped). **gap ratio = reproduced gap ÷ reported gap** (1 = we see exactly the author's gap).
Only real runs count (`summary.measured > 0`).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))          # stats/ (build_stats.py)
import pandas as pd
from build_stats import load_prs, per_field, plot, status_rows

df = load_prs()
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 40)
print(f"{len(df)} PRs reproduced here · {int((df.verdict=='reproduced').sum())} match the reported ratio (≥0.8 of it)")
df[['pr','field','hardware','headline','unit','reported_baseline','reported_pr_solution','reported_marker','reproduced_baseline','reproduced_pr_solution','reported_gap','reproduced_gap','gap_ratio','verdict','reliability','tests']]

In [ ]:
%matplotlib inline
pf = per_field(df)
fig = plot(df, pf, None)
pf

In [ ]:
pd.DataFrame(status_rows())[['pr','field','status','reason']]

## AI solver arms — solutions saved + CPU re-measured

One row per `(PR, arm)` solution under `data/pr_<n>/files/<arm>/`. `patch` = `files/<arm>.patch` emitted (easy remote apply), `remeasured` = `results.ai.<arm>.json` written by `pipeline/apply_ai_solutions.py` (CPU, same instrument). `ratio_arm_over_pr` > 1 = arm beats the PR's solution on the headline scenario (CPU number proves correctness, not the GPU speedup).

In [ ]:
from build_stats import ai_rows
ai = pd.DataFrame(ai_rows())
if len(ai):
    print(f"{len(ai)} solver solution(s) · {int((ai.status=='measured').sum())} CPU re-measured · "
          f"{int((ai.patch=='yes').sum())} with a ready-to-apply .patch")
    display(ai[['pr','arm','files','patch','remeasured','scenarios_measured','ratio_arm_over_pr','status']])
else:
    print("no solver solutions saved yet (data/pr_<n>/files/<arm>/)")